<div style='background:#000000;border-radius:10px;padding:26px 30px;border-top:6px solid #3777FF;border-bottom:3px solid #0EE071;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'>
<p style='color:#0EE071;margin:0;letter-spacing:2px;font-size:12px'>NOVA IMS · ENTERPRISE DATA SCIENCE BOOTCAMP 2026 · PROJETO D</p>
<h1 style='color:#FFFFFF;margin:6px 0 4px 0'>Telco Customer Churn · <span style='color:#3777FF'>Retention Management</span></h1>
<p style='color:#FFFFFF;margin:0;font-size:15px'>Notebook 02 — Modelação ensemble, avaliação contra o benchmark IBM e priorização por valor</p>
</div>

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>Índice — Estrutura do notebook</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Implementa os <b style='color:#3777FF'>próximos passos</b> definidos na secção 14 do notebook 01.
Pré-requisito: correr primeiro <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>01_preprocessing_eda.ipynb</code> (gera <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/processed/</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>preprocessing_config.joblib</code>).</p>
<ol style='margin:4px 0'>
<li>Setup do ambiente</li><li>Carregamento dos artefactos</li><li>Tuning dos modelos base</li>
<li>Ensembles e seleção do modelo</li><li>Satisfaction Score — com e sem</li><li>City com TargetEncoder</li>
<li>Avaliação final no holdout</li><li>Interpretação</li><li>Priorização P(churn) × CLTV</li>
<li>Exportação</li><li>Conclusões para o relatório</li></ol></div></div>

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>1 · Setup do ambiente</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Mesmo setup do notebook 01. O módulo novo <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>modelling</code> contém o tuning, os ensembles, a avaliação e a priorização.</p>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>N_ITER</code> controla o nº de combinações testadas por modelo no <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>RandomizedSearchCV</code> — reduzir para correr mais depressa.</p></div></div>

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import warnings
import joblib
import pandas as pd
from IPython.display import display
from sklearn.inspection import permutation_importance

from telco_churn import modelling, plots
from telco_churn.config import ARTIFACTS_DIR, DATA_PROCESSED, TAB_DIR, ensure_dirs
from telco_churn.style import apply_theme, style_table

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60)
SHOW = True          # False para correr sem mostrar gráficos (são sempre guardados)
N_ITER = 20          # combinações por modelo no RandomizedSearchCV

ensure_dirs()
apply_theme()
print("Raiz do projeto:", ROOT)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>2 · Carregamento dos artefactos</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Carrega <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>preprocessing_config.joblib</code> (listas de features e target) e os CSV de treino, teste e Joined.
Os CSV são lidos com <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>keep_default_na=False</code> para preservar o literal <i>None</i> em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Offer</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Internet Type</code>.</p>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>City</code> não foi exportada nos CSV de treino/teste (alta cardinalidade); é recuperada da base unificada por <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Customer ID</code> só para o teste da secção 6.</p></div></div>

In [ ]:
train, test, joined, cfg = modelling.load_processed()
NUM, CAT, TARGET = cfg["features_num"], cfg["features_cat"], cfg["target"]
FEATURES = NUM + CAT

X_train, y_train = train, train[TARGET]
X_test, y_test = test, test[TARGET]

display(style_table(pd.DataFrame({
    "linhas": [len(train), len(test), len(joined)],
    "taxa_churn": [y_train.mean(), y_test.mean(), None],
}, index=["treino", "teste (holdout)", "joined (scoring)"])))
print(f"{len(NUM)} numéricas + {len(CAT)} categóricas = {len(FEATURES)} features")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>3 · Tuning dos modelos base</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Cada modelo é um <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Pipeline([("prep", ...), ("clf", ...)])</code>: o pré-processador é ajustado <b style='color:#0EE071'>dentro de cada fold</b>, por isso não há fuga de informação na validação cruzada.</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>Logistic Regression</b> — pré-processador <i>linear</i> (com escala); referência simples e interpretável</li><li><b style='color:#3777FF'>Random Forest</b> e <b style='color:#3777FF'>Extra Trees</b> — bagging de árvores, pré-processador <i>trees</i></li><li><b style='color:#3777FF'>Gradient Boosting</b> — <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>HistGradientBoostingClassifier</code> (boosting do scikit-learn, equivalente a LightGBM)</li></ul>
<p>Procura aleatória com <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>StratifiedKFold</code> (5 folds) <b style='color:#0EE071'>só no treino</b>, métrica ROC AUC. O holdout não é usado até à secção 7.</p></div></div>

In [ ]:
X_tr = X_train[FEATURES]
best_params, tuning = modelling.tune_base_models(X_tr, y_train, NUM, CAT, n_iter=N_ITER)
tuning.to_csv(TAB_DIR / "10_tuning_modelos_base.csv")
display(style_table(tuning, highlight="cv_auc"))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>4 · Ensembles e seleção do modelo</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Os 4 modelos afinados são combinados de duas formas:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#0EE071'>Soft Voting</b> — média simples das probabilidades</li><li><b style='color:#0EE071'>Stacking</b> — um meta-modelo logístico aprende o peso de cada modelo a partir de previsões <i>out-of-fold</i></li></ul>
<p>Todos os modelos são comparados no mesmo 5-fold do treino: <b style='color:#3777FF'>AUC</b> (ordenação), <b style='color:#3777FF'>PR-AUC</b> (foco nos churners) e <b style='color:#3777FF'>Brier</b> (calibração — importante porque P(churn) multiplica o CLTV na priorização).
O modelo final é o de <b style='color:#0EE071'>maior AUC em CV</b>, escolhido antes de olhar para o holdout.</p></div></div>

In [ ]:
models = modelling.build_models(NUM, CAT, best_params)
cv_table = modelling.cv_compare(models, X_tr, y_train)
cv_table.to_csv(TAB_DIR / "11_cv_modelos.csv")
display(style_table(cv_table, precision=4, highlight="auc"))

auc_ibm_train = modelling.score_metrics(y_train, train["Churn Score"] / 100)["auc"]
plots.plot_cv_comparison(cv_table, benchmark_auc=auc_ibm_train, show=SHOW)

BEST = cv_table["auc"].idxmax()
print(f"Modelo selecionado (maior AUC em CV): {BEST}")
print(f"Referência — AUC Churn Score IBM no treino: {auc_ibm_train:.3f}")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>5 · Satisfaction Score — com e sem</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>No notebook 01, <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Satisfaction Score</code> 1–2 corresponde a 100% de churn: suspeita de ser registada no contexto da saída.
Treinamos o modelo selecionado <b style='color:#0EE071'>com e sem</b> a variável, no mesmo 5-fold, e documentamos a diferença.</p>
<p>Se o ganho for grande, é sinal de leakage: a versão com a variável <b style='color:#3777FF'>não serve para retenção preventiva</b>, porque o score só existe tarde demais.</p></div></div>

In [ ]:
NUM_SAT = NUM + ["Satisfaction Score"]
sat_models = {
    f"{BEST} — sem Satisfaction": models[BEST],
    f"{BEST} — com Satisfaction": modelling.build_models(NUM_SAT, CAT, best_params)[BEST],
}
sat_cv = pd.concat([
    modelling.cv_compare({k: m}, X_train[f], y_train)
    for (k, m), f in zip(sat_models.items(), [FEATURES, NUM_SAT + CAT])
])
display(style_table(sat_cv, precision=4, highlight="auc"))
print(f"Diferença de AUC (com − sem): {sat_cv['auc'].iloc[1] - sat_cv['auc'].iloc[0]:+.4f}")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>6 · City com TargetEncoder</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>City</code> tem ~1.100 valores distintos: one-hot é inviável. O <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>TargetEncoder</code> substitui cada cidade pela taxa de churn suavizada,
calculada com <i>cross-fitting</i> e <b style='color:#0EE071'>dentro do Pipeline</b> — em cada fold só vê o treino desse fold.</p>
<p>Só entra no modelo final se melhorar o AUC em CV de forma clara (mais do que o desvio-padrão entre folds).</p></div></div>

In [ ]:
city_model = modelling.build_models(NUM, CAT, best_params, te=["City"])[BEST]
city_cv = modelling.cv_compare({f"{BEST} + City (TargetEncoder)": city_model},
                               X_train[FEATURES + ["City"]], y_train)
variants = pd.concat([sat_cv, city_cv])
variants.to_csv(TAB_DIR / "12_variantes_satisfaction_city.csv")
display(style_table(variants, precision=4, highlight="auc"))

gain = city_cv["auc"].iloc[0] - cv_table.loc[BEST, "auc"]
USE_CITY = gain > cv_table.loc[BEST, "auc_std"]
print(f"Ganho de AUC com City: {gain:+.4f} -> {'incluir' if USE_CITY else 'não incluir'} no modelo final")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>7 · Avaliação final no holdout</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>O holdout (20%) é usado <b style='color:#0EE071'>uma única vez</b>: todos os modelos são treinados no treino completo e avaliados no teste.
O modelo final já foi escolhido na secção 4; os restantes aparecem só para transparência.</p>
<p>O <b style='color:#3777FF'>Churn Score IBM</b> é avaliado no mesmo holdout — é a fasquia a bater. O seu Brier é dado apenas como referência: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Churn Score / 100</code> não é uma probabilidade calibrada. A versão com Satisfaction Score é reportada à parte, para documentar o efeito do leakage.</p></div></div>

In [ ]:
FINAL_FEATURES = FEATURES + (["City"] if USE_CITY else [])
final_models = dict(models)
if USE_CITY:
    final_models[BEST] = city_model

probas = {}
for name, model in final_models.items():
    cols = FINAL_FEATURES if name == BEST else FEATURES
    probas[name] = model.fit(X_train[cols], y_train).predict_proba(X_test[cols])[:, 1]

sat_model = sat_models[f"{BEST} — com Satisfaction"]
p_sat = sat_model.fit(X_train[NUM_SAT + CAT], y_train).predict_proba(X_test[NUM_SAT + CAT])[:, 1]

holdout = modelling.holdout_report(probas | {f"{BEST} + Satisfaction (leakage)": p_sat},
                                   y_test, test["Churn Score"])
holdout.to_csv(TAB_DIR / "13_holdout.csv")
display(style_table(holdout, precision=4, highlight="auc"))

final_model = final_models[BEST]
p_test = probas[BEST]
plots.plot_roc_calibration(y_test, {BEST: p_test, "Churn Score IBM": test["Churn Score"] / 100,
                                    "Logistic Regression": probas["Logistic Regression"]}, show=SHOW)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>8 · Interpretação</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p><b style='color:#0EE071'>Permutation importance</b> no holdout: quanto cai o AUC quando os valores de uma variável são baralhados.
Funciona para qualquer modelo (incluindo ensembles) e ao nível das variáveis originais, não das colunas one-hot — mais útil para explicar ao Head of Customer Retention.</p>
<p>Barras <b style='color:#3777FF'>azuis</b> = numéricas, <b style='color:#0EE071'>verdes</b> = categóricas.</p></div></div>

In [ ]:
pi = permutation_importance(final_model, X_test[FINAL_FEATURES], y_test, scoring="roc_auc",
                            n_repeats=5, random_state=0, n_jobs=-1)
importance = (pd.Series(pi.importances_mean, index=FINAL_FEATURES)
              .sort_values(ascending=False).clip(lower=0))
importance.to_frame("queda_auc").to_csv(TAB_DIR / "14_importancia_permutacao.csv")
plots.plot_ranking(importance.head(20), f"Permutation importance — {BEST} (queda de AUC, holdout)",
                   "27_importancia_permutacao", highlight=set(CAT) | {"City"}, show=SHOW)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>9 · Priorização: P(churn) × CLTV</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>O objetivo não é só prever churn, é decidir <b style='color:#3777FF'>quem contactar primeiro</b>. Cada cliente recebe uma
<b style='color:#0EE071'>perda esperada = P(churn) × CLTV</b> e a lista de intervenção é ordenada por esse valor.</p>
<p>A <b style='color:#0EE071'>curva de captura</b> mede, no holdout, que % do CLTV dos clientes que de facto saíram fica coberta ao contactar os top-k% da lista.
Compara quatro ordenações: o nosso P × CLTV, só P(churn), o Churn Score IBM × CLTV e o Churn Score IBM sozinho.</p></div></div>

In [ ]:
orderings = {
    f"{BEST}: P × CLTV": p_test * test["CLTV"],
    f"{BEST}: só P(churn)": p_test,
    "IBM: Churn Score × CLTV": test["Churn Score"] * test["CLTV"],
    "IBM: só Churn Score": test["Churn Score"],
}
curve = modelling.capture_curve(y_test, test["CLTV"], orderings)
capture = modelling.capture_at(curve, cuts=(0.1, 0.2, 0.3))
capture.to_csv(TAB_DIR / "15_captura_valor.csv")
display(style_table(capture, precision=3))
plots.plot_capture_curve(curve, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>9.1 · Lista de intervenção</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Lista ordenada para o holdout (onde podemos verificar o resultado) e para os clientes <b style='color:#3777FF'>Joined</b>, que ficaram fora do treino.
O mapa mostra onde ficam os top-N: o P × CLTV troca alguns clientes de risco alto mas pouco valor por clientes de valor alto com risco moderado.</p>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>TOP_N</code> é o orçamento de contactos — ajustar ao que a equipa de retenção consegue tratar.</p>
<p><b style='color:#0EE071'>Atenção aos Joined:</b> são clientes com poucos meses de casa. Na população de treino, tenure baixa está fortemente associada a churn,
por isso o modelo tende a dar-lhes P(churn) muito alta — é <b style='color:#3777FF'>extrapolação</b>, não um resultado validado.
A lista dos Joined deve ser lida como ordenação relativa e não como probabilidade calibrada.</p></div></div>

In [ ]:
TOP_N = 200
prio_test = modelling.priority_list(test, p_test)
prio_test = prio_test.merge(test[["Customer ID", TARGET]], on="Customer ID")
plots.plot_priority_map(prio_test, TOP_N, show=SHOW)

by_decile = prio_test.groupby("decil").agg(clientes=("rank", "size"), p_churn_media=("p_churn", "mean"),
                                           churn_real=(TARGET, "mean"), cltv_medio=("CLTV", "mean"),
                                           perda_esperada=("perda_esperada", "sum"))
display(style_table(by_decile, precision=3, highlight="churn_real"))
display(style_table(prio_test.head(15), precision=3))

In [ ]:
# Clientes Joined: novos clientes, sem resultado conhecido -> só scoring (ver nota acima: extrapolação)
p_joined = final_model.predict_proba(joined[FINAL_FEATURES])[:, 1]
prio_joined = modelling.priority_list(joined, p_joined)
print(f"Joined: {len(prio_joined)} clientes | P(churn) média {p_joined.mean():.1%}")
display(style_table(prio_joined.head(15), precision=3))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>10 · Exportação</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Ficheiros gerados (ignorados pelo Git):</p>
<ul style='margin:6px 0 2px 0'><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/artifacts/final_model.joblib</code> — modelo final treinado, lista de features e métricas</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/processed/prioridade_holdout.csv</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>prioridade_joined.csv</code> — listas de intervenção ordenadas</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/tables/10_*</code> a <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>15_*</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/figures/2*_</code> — tabelas e gráficos desta fase</li></ul></div></div>

In [ ]:
bundle = {"model": final_model, "name": BEST, "features": FINAL_FEATURES, "target": TARGET,
          "best_params": best_params, "holdout": holdout.loc[BEST].to_dict()}
paths = [ARTIFACTS_DIR / "final_model.joblib",
         DATA_PROCESSED / "prioridade_holdout.csv", DATA_PROCESSED / "prioridade_joined.csv"]
joblib.dump(bundle, paths[0])
prio_test.to_csv(paths[1], index=False)
prio_joined.to_csv(paths[2], index=False)
for p in paths:
    print("✔", p.relative_to(ROOT))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>11 · Conclusões para o relatório</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>A célula seguinte resume os números-chave a partir das tabelas acima, para transcrever para o relatório:</p>
<ul style='margin:6px 0 2px 0'><li>desempenho do modelo final vs <b style='color:#3777FF'>Churn Score IBM</b> no mesmo holdout</li><li>efeito do <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Satisfaction Score</code> (leakage) e decisão sobre <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>City</code></li><li>ganho de valor captado com a priorização <b style='color:#0EE071'>P(churn) × CLTV</b></li></ul></div></div>

In [ ]:
h = holdout
ibm = "Churn Score IBM (benchmark)"
print(f"Modelo final: {BEST}")
print(f"  AUC holdout {h.loc[BEST, 'auc']:.3f} vs IBM {h.loc[ibm, 'auc']:.3f} "
      f"({h.loc[BEST, 'auc'] - h.loc[ibm, 'auc']:+.3f})")
print(f"  PR-AUC {h.loc[BEST, 'pr_auc']:.3f} | Brier {h.loc[BEST, 'brier']:.3f}")
print(f"Satisfaction Score: AUC holdout com a variável = "
      f"{h.loc[f'{BEST} + Satisfaction (leakage)', 'auc']:.3f} (fora do modelo final)")
print(f"City (TargetEncoder): {'incluída' if USE_CITY else 'não incluída'} (ganho CV {gain:+.4f})")
top20 = capture.loc[:, "top 20%"]
print(f"Contactando 20% dos clientes: P × CLTV capta {top20.iloc[0]:.1%} do CLTV perdido "
      f"vs {top20.loc['IBM: só Churn Score']:.1%} com o Churn Score IBM")